In [1]:
import requests
import pandas as pd

from bs4 import BeautifulSoup

import sqlite3

import time

import random

import logging

In [2]:
import os

# Create folders
os.makedirs("scraper", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# Create empty files
files = [
    "scraper/__init__.py",
    "scraper/config.py",
    "scraper/parser.py",
    "scraper/scraper.py",
    "scraper/utils.py"
]

for file in files:
    if not os.path.exists(file):
        open(file, "w").close()

print("Project structure created successfully!")

Project structure created successfully!


# Logging

In [3]:
import os
import logging

# Create logs directory if it doesn't exist
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/scraper.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Logger Started")

# User Agent

In [4]:
HEADERS = {

    "User-Agent":

    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/137 Safari/537.36"

}

In [5]:
# Choose Website
BASE_URL = "https://realpython.github.io/fake-jobs/"

# Download HTML

In [6]:
response = requests.get(

    BASE_URL,

    headers=HEADERS

)

print(response.status_code)

200


# Parse HTML

In [7]:
soup = BeautifulSoup(

    response.text,

    "html.parser"

)

# Inspect HTML

In [8]:
print(soup.title.text)

Fake Python


In [9]:
print(soup.prettify()[:1000])

<!DOCTYPE html>
<html>
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <title>
   Fake Python
  </title>
  <link href="https://cdn.jsdelivr.net/npm/bulma@0.9.2/css/bulma.min.css" rel="stylesheet"/>
 </head>
 <body>
  <section class="section">
   <div class="container mb-5">
    <h1 class="title is-1">
     Fake Python
    </h1>
    <p class="subtitle is-3">
     Fake Jobs for Your Web Scraping Journey
    </p>
   </div>
   <div class="container">
    <div class="columns is-multiline" id="ResultsContainer">
     <div class="column is-half">
      <div class="card">
       <div class="card-content">
        <div class="media">
         <div class="media-left">
          <figure class="image is-48x48">
           <img alt="Real Python Logo" src="https://files.realpython.com/media/real-python-logo-thumbnail.7f0db70c2ed2.jpg?__no_cf_polish=1"/>
          </figure>
         </div>
         <div class="media-content">
          <h2 c

# Find Job Cards

In [10]:
job_cards = soup.find_all(

    "div",

    class_="card-content"

)

print(len(job_cards))

100


# Inspect First Job Card

In [11]:
print(job_cards[0].prettify())

<div class="card-content">
 <div class="media">
  <div class="media-left">
   <figure class="image is-48x48">
    <img alt="Real Python Logo" src="https://files.realpython.com/media/real-python-logo-thumbnail.7f0db70c2ed2.jpg?__no_cf_polish=1"/>
   </figure>
  </div>
  <div class="media-content">
   <h2 class="title is-5">
    Senior Python Developer
   </h2>
   <h3 class="subtitle is-6 company">
    Payne, Roberts and Davis
   </h3>
  </div>
 </div>
 <div class="content">
  <p class="location">
   Stewartbury, AA
  </p>
  <p class="is-small has-text-grey">
   <time datetime="2021-04-08">
    2021-04-08
   </time>
  </p>
 </div>
 <footer class="card-footer">
  <a class="card-footer-item" href="https://www.realpython.com" target="_blank">
   Learn
  </a>
  <a class="card-footer-item" href="https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html" target="_blank">
   Apply
  </a>
 </footer>
</div>



# Extract Job Title

In [12]:
title = job_cards[0].find(

    "h2",

    class_="title"

).text.strip()

print(title)

Senior Python Developer


# Extract Company

In [13]:
company = job_cards[0].find(

    "h3",

    class_="subtitle"

).text.strip()

print(company)

Payne, Roberts and Davis


# Extract Location

In [14]:
location = job_cards[0].find(

    "p",

    class_="location"

).text.strip()

print(location)

Stewartbury, AA


# Extract Apply Link

In [15]:
link = job_cards[0].find("a")

print(link["href"])

https://www.realpython.com


# Extract All Jobs

In [16]:
jobs = []

for card in job_cards:

    title = card.find(

        "h2",

        class_="title"

    ).text.strip()

    company = card.find(

        "h3",

        class_="subtitle"

    ).text.strip()

    location = card.find(

        "p",

        class_="location"

    ).text.strip()

    apply = card.find("a")["href"]

    jobs.append({

        "Title": title,

        "Company": company,

        "Location": location,

        "Apply Link": apply

    })

print(len(jobs))

100


# Create DataFrame

In [17]:
df = pd.DataFrame(jobs)

print(df.head())

                     Title                     Company              Location  \
0  Senior Python Developer    Payne, Roberts and Davis       Stewartbury, AA   
1          Energy engineer            Vasquez-Davidson  Christopherville, AA   
2          Legal executive  Jackson, Chambers and Levy   Port Ericaburgh, AA   
3   Fitness centre manager              Savage-Bradley     East Seanview, AP   
4          Product manager                 Ramirez Inc   North Jamieview, AP   

                   Apply Link  
0  https://www.realpython.com  
1  https://www.realpython.com  
2  https://www.realpython.com  
3  https://www.realpython.com  
4  https://www.realpython.com  


# Save CSV

In [18]:
df.to_csv(

    "data/jobs.csv",

    index=False

)

In [19]:
# Save Excel
df.to_excel(

    "data/jobs.xlsx",

    index=False

)

In [20]:
# scraper/config.py

BASE_URL = "https://realpython.github.io/fake-jobs/"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

TIMEOUT = 10

MIN_DELAY = 2
MAX_DELAY = 5

In [21]:
# scraper/utils.py

import random
import time

def random_delay(min_delay, max_delay):
    delay = random.uniform(min_delay, max_delay)
    print(f"Waiting {delay:.2f} seconds...")
    time.sleep(delay)

In [22]:
import logging

logging.basicConfig(
    filename="logs/scraper.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logging.info("Application Started")

In [26]:
BASE_URL = "https://www.indeed.com/jobs"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

TIMEOUT = 10

In [27]:
import requests

from scraper.config import BASE_URL
from scraper.config import HEADERS
from scraper.config import TIMEOUT


class JobScraper:

    def __init__(self):
        self.url = BASE_URL

    def download_page(self):

        response = requests.get(
            self.url,
            headers=HEADERS,
            timeout=TIMEOUT
        )

        response.raise_for_status()

        return response.text

ImportError: cannot import name 'BASE_URL' from 'scraper.config' (C:\Users\2005a\Data Science\Project\web scrapping\JobPortalAnalyzer\scraper\config.py)

In [ ]:
import os
import sys

# Go one level up to the project root
sys.path.append(os.path.abspath(".."))

In [ ]:
import os
import sys

print("Current Working Directory:")
print(os.getcwd())

print("\nFiles in Current Directory:")
print(os.listdir())

print("\nPython Path:")
print(sys.path)

In [ ]:
import os

for root, dirs, files in os.walk("."):
    print(root)
    for f in files:
        print("   ", f)

In [ ]:
def download_page(self):

    attempts = 3

    for attempt in range(attempts):

        try:

            response = requests.get(
                self.url,
                headers=HEADERS,
                timeout=TIMEOUT
            )

            response.raise_for_status()

            return response.text

        except Exception as e:

            print(f"Attempt {attempt+1} Failed")

            print(e)

    return None